# Day 6: Warning features for future lice Breaches
## Goal
* Build time-series features from historical lice observations that may help predict future regulatory breaches.

In [81]:
# check data frame
# Load the clean data sets
import pandas as pd
from pathlib import Path

clean_file = (
    Path.home() 
    /"Documents"
    /"Kazi_Academic"
    /"Projects"
    /"Aquaculture"
    /"fish-health-analytics"
    /"data"
    /"processed"
    /"barentswatch_lice_2025_clean.csv"
)
df = pd.read_csv(clean_file)
df. shape

/tmp/ipykernel_116561/2270916884.py:17: DtypeWarning: Columns (0: weekly_lice_limit) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(clean_file)


(55718, 22)

In [82]:
df.columns.tolist()

['week',
 'year',
 'locality_id',
 'locality_name',
 'adult_female_lice',
 'mobile_lice',
 'sessile_lice',
 'probably_without_fish',
 'lice_counted',
 'municipality_id',
 'municipality',
 'county_id',
 'county',
 'latitude',
 'longitude',
 'weekly_lice_limit',
 'above_weekly_lice_limit',
 'sea_temperature_c',
 'production_area_id',
 'production_area',
 'weekly_lice_limit_numeric',
 'compliance_code']

In [83]:
# copy the data into df_lice
df_lice = df.copy()

In [84]:
df


,week,year,locality_id,locality_name,adult_female_lice,mobile_lice,sessile_lice,probably_without_fish,lice_counted,municipality_id,...,county,latitude,longitude,weekly_lice_limit,above_weekly_lice_limit,sea_temperature_c,production_area_id,production_area,weekly_lice_limit_numeric,compliance_code
0,52,2025,15196,Aga Ø,0.30,2.87,0.12,Nei,Ja,4613.0,...,Vestland,59.845917,5.260750,0.5,Nei,8.40,3.0,Karmøy til Sotra,0.5,0.0
1,52,2025,12067,Aldalen,NaN,NaN,NaN,Ja,Nei,4624.0,...,Vestland,60.250015,5.571783,0.5,NaN,NaN,3.0,Karmøy til Sotra,0.5,NaN
2,52,2025,12982,Aldeøyna,NaN,NaN,NaN,Ja,Nei,4645.0,...,Vestland,61.310550,4.767783,0.5,NaN,NaN,4.0,Nordhordaland til Stadt,0.5,NaN
3,52,2025,11756,Allersholmen,NaN,NaN,NaN,Ja,Nei,4632.0,...,Vestland,60.774868,4.853533,0.5,NaN,NaN,4.0,Nordhordaland til Stadt,0.5,NaN
4,52,2025,45078,Almbakkevika,NaN,NaN,NaN,Ja,Nei,4602.0,...,Vestland,61.620335,5.271967,0.5,NaN,NaN,4.0,Nordhordaland til Stadt,0.5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55713,1,2025,45077,Ørnhaugneset,0.40,0.93,0.00,Nei,Ja,1875.0,...,Nordland,67.980250,15.587800,0.5,Nei,4.76,9.0,Vestfjorden og Vesterålen,0.5,0.0
55714,1,2025,12394,Ørnøya,NaN,NaN,NaN,Ja,Nei,5014.0,...,Trøndelag,63.761868,8.445183,0.5,NaN,NaN,6.0,Nordmøre og Sør-Trøndelag,0.5,NaN
55715,1,2025,24696,Ørnøya II,NaN,NaN,NaN,Ja,Nei,5014.0,...,Trøndelag,63.758484,8.450367,0.5,NaN,NaN,6.0,Nordmøre og Sør-Trøndelag,0.5,NaN
55716,1,2025,11378,Øvergården,NaN,NaN,NaN,Ja,Nei,5503.0,...,Troms,68.981670,16.534666,0.5,NaN,NaN,10.0,Andøya til Senja,0.5,NaN


In [85]:
# Identify exact locality, time, and lice-count columns.
df_lice.columns.tolist()

['week',
 'year',
 'locality_id',
 'locality_name',
 'adult_female_lice',
 'mobile_lice',
 'sessile_lice',
 'probably_without_fish',
 'lice_counted',
 'municipality_id',
 'municipality',
 'county_id',
 'county',
 'latitude',
 'longitude',
 'weekly_lice_limit',
 'above_weekly_lice_limit',
 'sea_temperature_c',
 'production_area_id',
 'production_area',
 'weekly_lice_limit_numeric',
 'compliance_code']

In [86]:
# each locality has only one observation per year-week.
df_lice[["locality_id", "year", "week"]].duplicated().sum()


np.int64(0)

In [87]:
df_lice["compliance_code"].value_counts(dropna=False)

compliance_code
0.0    29231
NaN    25421
1.0     1066
Name: count, dtype: int64

In [88]:
# sort each locality chronologically.
df_lice = df_lice.sort_values(
    ["locality_id",  "year", "week"]
    ).copy()

In [89]:
# Inspect one locality
df_lice[
    df_lice["locality_id"] == 15196
][
    ["locality_id", "locality_name", "year", "week", "adult_female_lice"]
].head(15)

,locality_id,locality_name,year,week,adult_female_lice
54649,15196,Aga Ø,2025,1,NaN
53580,15196,Aga Ø,2025,2,NaN
52511,15196,Aga Ø,2025,3,NaN
51442,15196,Aga Ø,2025,4,NaN
50373,15196,Aga Ø,2025,5,NaN
49303,15196,Aga Ø,2025,6,NaN
48233,15196,Aga Ø,2025,7,NaN
47161,15196,Aga Ø,2025,8,NaN
46090,15196,Aga Ø,2025,9,NaN
45019,15196,Aga Ø,2025,10,NaN


In [90]:
# Step 1: Select one locality
one_locality = df_lice[df_lice["locality_id"] == 15196]

# Step 2: Select only the columns we want to see
columns_to_show = [
    "locality_id",
    "locality_name",
    "year",
    "week",
    "adult_female_lice"
]

one_locality = one_locality[columns_to_show]

# Step 3: Show the first 15 rows
one_locality.head(15)

,locality_id,locality_name,year,week,adult_female_lice
54649,15196,Aga Ø,2025,1,NaN
53580,15196,Aga Ø,2025,2,NaN
52511,15196,Aga Ø,2025,3,NaN
51442,15196,Aga Ø,2025,4,NaN
50373,15196,Aga Ø,2025,5,NaN
49303,15196,Aga Ø,2025,6,NaN
48233,15196,Aga Ø,2025,7,NaN
47161,15196,Aga Ø,2025,8,NaN
46090,15196,Aga Ø,2025,9,NaN
45019,15196,Aga Ø,2025,10,NaN


In [91]:
# Before we build lag_1, let's inspect when this locality actually starts having lice data.
# Keep only rows where adult female lice was measured
aga_with_lice = one_locality[one_locality["adult_female_lice"].notna()]


In [92]:
# Show the first 10 measured weeks
aga_with_lice.head(10)

,locality_id,locality_name,year,week,adult_female_lice
31100,15196,Aga Ø,2025,23,0.00
30029,15196,Aga Ø,2025,24,0.00
28960,15196,Aga Ø,2025,25,0.00
27889,15196,Aga Ø,2025,26,0.03
26816,15196,Aga Ø,2025,27,0.02
25743,15196,Aga Ø,2025,28,0.10
24669,15196,Aga Ø,2025,29,0.23
23595,15196,Aga Ø,2025,30,0.30
22521,15196,Aga Ø,2025,31,0.26
21447,15196,Aga Ø,2025,32,0.20


In [93]:
# create lag_1
# Step 1: Group the data by locality
grouped_localities = df_lice.groupby("locality_id")

# Step 2: Take the adult female lice column
lice_values = grouped_localities["adult_female_lice"]

# Step 3: Shift the values by 1 row
previous_lice = lice_values.shift(1)

# Step 4: Save it as a new column
df_lice["adult_female_lice_lag_1"] = previous_lice

In [94]:
# Create lag2
# Step 1: Group the data by locality
grouped_localities = df_lice.groupby("locality_id")

# Step 2: Take the adult female lice values
lice_values = grouped_localities["adult_female_lice"]

# Step 3: Shift the values by 2 rows
two_weeks_back = lice_values.shift(2)

# Step 4: Save as a new column
df_lice["adult_female_lice_lag_2"] = two_weeks_back

In [95]:
# lag columns for Aga Ø
aga = df_lice[df_lice["locality_id"] == 15196]

columns_to_show = [
    "year",
    "week",
    "adult_female_lice",
    "adult_female_lice_lag_1",
    "adult_female_lice_lag_2"
]

aga[columns_to_show].iloc[20:35]

,year,week,adult_female_lice,adult_female_lice_lag_1,adult_female_lice_lag_2
33242,2025,21,NaN,NaN,NaN
32171,2025,22,NaN,NaN,NaN
31100,2025,23,0.00,NaN,NaN
30029,2025,24,0.00,0.00,NaN
28960,2025,25,0.00,0.00,0.00
27889,2025,26,0.03,0.00,0.00
26816,2025,27,0.02,0.03,0.00
25743,2025,28,0.10,0.02,0.03
24669,2025,29,0.23,0.10,0.02
23595,2025,30,0.30,0.23,0.10


In [96]:
# Weekly lice change

# We want: current lice - previous lice
# Step 1: Take current lice values
current_lice = df_lice["adult_female_lice"]

# Step 2: Take previous week's lice values
previous_lice = df_lice["adult_female_lice_lag_1"]

# Step 3: Calculate the change
lice_change = current_lice - previous_lice

# Step 4: Save it as a new column
df_lice["adult_female_lice_change"] = lice_change

In [97]:
columns_to_show = [
    "year",
    "week",
    "adult_female_lice",
    "adult_female_lice_lag_1",
    "adult_female_lice_change"
]

aga = df_lice[df_lice["locality_id"] == 15196]

aga[columns_to_show].iloc[22:35]

,year,week,adult_female_lice,adult_female_lice_lag_1,adult_female_lice_change
31100,2025,23,0.00,NaN,NaN
30029,2025,24,0.00,0.00,0.00
28960,2025,25,0.00,0.00,0.00
27889,2025,26,0.03,0.00,0.03
26816,2025,27,0.02,0.03,-0.01
25743,2025,28,0.10,0.02,0.08
24669,2025,29,0.23,0.10,0.13
23595,2025,30,0.30,0.23,0.07
22521,2025,31,0.26,0.30,-0.04
21447,2025,32,0.20,0.26,-0.06


In [98]:
# Next: change over two weeks
# Step 1: Current lice value
current_lice = df_lice["adult_female_lice"]

# Step 2: Lice value from two weeks ago
lice_two_weeks_ago = df_lice["adult_female_lice_lag_2"]

# Step 3: Calculate the two-week change
two_week_change = current_lice - lice_two_weeks_ago

# Step 4: Save it
df_lice["adult_female_lice_change_2w"] = two_week_change

In [99]:
columns_to_show = [
    "year",
    "week",
    "adult_female_lice",
    "adult_female_lice_lag_2",
    "adult_female_lice_change_2w"
]

aga = df_lice[df_lice["locality_id"] == 15196]

aga[columns_to_show].iloc[22:35]

,year,week,adult_female_lice,adult_female_lice_lag_2,adult_female_lice_change_2w
31100,2025,23,0.00,NaN,NaN
30029,2025,24,0.00,NaN,NaN
28960,2025,25,0.00,0.00,0.00
27889,2025,26,0.03,0.00,0.03
26816,2025,27,0.02,0.00,0.02
25743,2025,28,0.10,0.03,0.07
24669,2025,29,0.23,0.02,0.21
23595,2025,30,0.30,0.10,0.20
22521,2025,31,0.26,0.23,0.03
21447,2025,32,0.20,0.30,-0.10


In [100]:
aga[
    ["week", "adult_female_lice", "adult_female_lice_lag_2",
     "adult_female_lice_change_2w"]
].tail()

,week,adult_female_lice,adult_female_lice_lag_2,adult_female_lice_change_2w
4275,48,0.06,0.03,0.03
3206,49,0.09,0.05,0.04
2137,50,0.11,0.06,0.05
1069,51,0.14,0.09,0.05
0,52,0.30,0.11,0.19


In [101]:
# Next feature: 3-week rolling mean
# Step 1: Group by locality
grouped_localities = df_lice.groupby("locality_id")

# Step 2: Select adult female lice
lice_values = grouped_localities["adult_female_lice"]

# Step 3: Calculate the 3-week rolling average
rolling_mean_3w = lice_values.transform(
    lambda x: x.rolling(window=3, min_periods=3).mean()
)

# Step 4: Save it as a new column
df_lice["adult_female_lice_mean_3w"] = rolling_mean_3w

In [102]:
"adult_female_lice_mean_3w" in df_lice.columns

True

In [103]:
# Group lice values by locality
lice_by_locality = df_lice.groupby("locality_id")["adult_female_lice"]

# Calculate 3-week average
mean_3w = lice_by_locality.transform(
    lambda values: values.rolling(3, min_periods=3).mean()
)

# Add the new column
df_lice["adult_female_lice_mean_3w"] = mean_3w

In [104]:
"adult_female_lice_mean_3w" in df_lice.columns

True

In [105]:
aga = df_lice[df_lice["locality_id"] == 15196]

In [106]:
columns_to_show = [
    "week",
    "adult_female_lice",
    "adult_female_lice_mean_3w"
]

aga[columns_to_show].iloc[22:35]

,week,adult_female_lice,adult_female_lice_mean_3w
31100,23,0.00,NaN
30029,24,0.00,NaN
28960,25,0.00,0.000000
27889,26,0.03,0.010000
26816,27,0.02,0.016667
25743,28,0.10,0.050000
24669,29,0.23,0.116667
23595,30,0.30,0.210000
22521,31,0.26,0.263333
21447,32,0.20,0.253333


In [107]:
# Next feature: distance to the lice limit

# Now we calculate:
# Step 1: Take the weekly lice limit
lice_limit = df_lice["weekly_lice_limit_numeric"]

# Step 2: Take the current adult female lice value
current_lice = df_lice["adult_female_lice"]

# Step 3: Calculate distance from the limit
distance_to_limit = lice_limit - current_lice

# Step 4: Save it as a new column
df_lice["distance_to_limit"] = distance_to_limit

In [108]:
#check Aga Ø with
aga = df_lice[df_lice["locality_id"] == 15196]

columns_to_show = [
    "week",
    "adult_female_lice",
    "weekly_lice_limit_numeric",
    "distance_to_limit"
]

aga[columns_to_show].iloc[22:35]

,week,adult_female_lice,weekly_lice_limit_numeric,distance_to_limit
31100,23,0.00,0.5,0.50
30029,24,0.00,0.5,0.50
28960,25,0.00,0.5,0.50
27889,26,0.03,0.5,0.47
26816,27,0.02,0.5,0.48
25743,28,0.10,0.5,0.40
24669,29,0.23,0.5,0.27
23595,30,0.30,0.5,0.20
22521,31,0.26,0.5,0.24
21447,32,0.20,0.5,0.30


In [109]:
# ratio to the lice limit.
# Step 1: Take current lice
current_lice = df_lice["adult_female_lice"]

# Step 2: Take the weekly limit
lice_limit = df_lice["weekly_lice_limit_numeric"]

# Step 3: Calculate the ratio
ratio_to_limit = current_lice / lice_limit

# Step 4: Save it
df_lice["ratio_to_limit"] = ratio_to_limit

In [110]:
# Update Aga Ø from df_lice
aga = df_lice[df_lice["locality_id"] == 15196]

# Columns to show
columns_to_show = [
    "week",
    "adult_female_lice",
    "weekly_lice_limit_numeric",
    "ratio_to_limit"
]

aga[columns_to_show].iloc[22:35]

,week,adult_female_lice,weekly_lice_limit_numeric,ratio_to_limit
31100,23,0.00,0.5,0.00
30029,24,0.00,0.5,0.00
28960,25,0.00,0.5,0.00
27889,26,0.03,0.5,0.06
26816,27,0.02,0.5,0.04
25743,28,0.10,0.5,0.20
24669,29,0.23,0.5,0.46
23595,30,0.30,0.5,0.60
22521,31,0.26,0.5,0.52
21447,32,0.20,0.5,0.40


In [111]:
# Previous breach status

# Did this locality breach the limit in the previous week?
# Step 1: Group by locality
grouped_localities = df_lice.groupby("locality_id")

# Step 2: Take compliance status
compliance_values = grouped_localities["compliance_code"]

# Step 3: Shift by 1 week
previous_breach = compliance_values.shift(1)

# Step 4: Save it
df_lice["previous_breach"] = previous_breach

In [112]:
# check Aga Ø
# Refresh Aga Ø
aga = df_lice[df_lice["locality_id"] == 15196]

columns_to_show = [
    "week",
    "adult_female_lice",
    "compliance_code",
    "previous_breach"
]

aga[columns_to_show].iloc[22:40]

,week,adult_female_lice,compliance_code,previous_breach
31100,23,0.00,0.0,NaN
30029,24,0.00,0.0,0.0
28960,25,0.00,0.0,0.0
27889,26,0.03,0.0,0.0
26816,27,0.02,0.0,0.0
25743,28,0.10,0.0,0.0
24669,29,0.23,0.0,0.0
23595,30,0.30,0.0,0.0
22521,31,0.26,0.0,0.0
21447,32,0.20,0.0,0.0


In [113]:
# create the prediction target: whether the locality breaches the lice limit in the following week.
# Step 1: Group by locality
grouped_localities = df_lice.groupby("locality_id")

# Step 2: Take the compliance status
compliance_values = grouped_localities["compliance_code"]

# Step 3: Take next week's compliance status
next_week_breach = compliance_values.shift(-1)

# Step 4: Save it as the prediction target
df_lice["future_breach"] = next_week_breach

In [114]:
# check wih Aga
# Refresh Aga Ø
aga = df_lice[df_lice["locality_id"] == 15196]

columns_to_show = [
    "week",
    "adult_female_lice",
    "compliance_code",
    "future_breach"
]

aga[columns_to_show].iloc[22:40]

,week,adult_female_lice,compliance_code,future_breach
31100,23,0.00,0.0,0.0
30029,24,0.00,0.0,0.0
28960,25,0.00,0.0,0.0
27889,26,0.03,0.0,0.0
26816,27,0.02,0.0,0.0
25743,28,0.10,0.0,0.0
24669,29,0.23,0.0,0.0
23595,30,0.30,0.0,0.0
22521,31,0.26,0.0,0.0
21447,32,0.20,0.0,0.0


In [115]:
# Count future breach values
df_lice["future_breach"].value_counts(dropna=False)

future_breach
0.0    28646
NaN    26018
1.0     1054
Name: count, dtype: int64

In [116]:
# Keep only rows where next week's breach status is known
df_valid = df_lice[
    df_lice["future_breach"].notna()
].copy()

In [117]:
# Percentage among valid future outcomes only
df_valid["future_breach"].value_counts(
    normalize=True
) * 100

future_breach
0.0    96.451178
1.0     3.548822
Name: proportion, dtype: float64

In [118]:
# Now let's ask whether our engineered warning features actually look different before a future breach.
# Features we created
warning_features = [
    "adult_female_lice",
    "adult_female_lice_lag_1",
    "adult_female_lice_change",
    "adult_female_lice_change_2w",
    "adult_female_lice_mean_3w",
    "distance_to_limit",
    "ratio_to_limit",
    "previous_breach"
]

# Compare average feature values
df_valid.groupby("future_breach")[warning_features].mean().round(3)


,adult_female_lice,adult_female_lice_lag_1,adult_female_lice_change,adult_female_lice_change_2w,adult_female_lice_mean_3w,distance_to_limit,ratio_to_limit,previous_breach
future_breach,,,,,,,,
0.0,0.157,0.162,-0.003,-0.001,0.164,0.307,0.336,0.027
1.0,0.677,0.462,0.217,0.306,0.506,-0.205,1.427,0.231


In [119]:
warning_features = [
    "adult_female_lice",
    "adult_female_lice_lag_1",
    "adult_female_lice_change",
    "adult_female_lice_change_2w",
    "adult_female_lice_mean_3w",
    "distance_to_limit",
    "ratio_to_limit",
    "previous_breach"
]

df_valid.groupby("future_breach")[warning_features].mean().round(3)

,adult_female_lice,adult_female_lice_lag_1,adult_female_lice_change,adult_female_lice_change_2w,adult_female_lice_mean_3w,distance_to_limit,ratio_to_limit,previous_breach
future_breach,,,,,,,,
0.0,0.157,0.162,-0.003,-0.001,0.164,0.307,0.336,0.027
1.0,0.677,0.462,0.217,0.306,0.506,-0.205,1.427,0.231


In [120]:
# Where we want to save the feature dataset
feature_file = (
    Path.home()
    / "Documents"
    / "Kazi_Academic"
    / "Projects"
    / "Aquaculture"
    / "fish-health-analytics"
    / "data"
    / "processed"
    / "barentswatch_lice_2025_features.csv"
)

# Save the Day 6 dataframe
df_lice.to_csv(feature_file, index=False)